# Automated EDA: From First Principles

## Why Explore Data Before Modeling?

Exploratory Data Analysis (EDA) is the process of examining data to understand its structure, quality, distributions, and relationships before deciding how to model it. Automated EDA uses software to generate summaries and visualizations quickly; it does not automatically clean the data or decide which model is appropriate.

A model only sees the representation it is given. A date stored as text, an impossible measurement, a duplicated row, a target copied into a feature, or missing values concentrated in one group can distort training and evaluation. EDA helps surface these risks while there is still time to investigate them.

The goal is not to make every column look statistically tidy. The goal is to learn enough about how the data was collected and what it represents to make defensible decisions.

## Manual and Automated EDA

Manual EDA means asking a question and inspecting the relevant tables, calculations, or plots. It gives the analyst control over context, assumptions, and edge cases, but takes time and can miss patterns when the dataset is wide or unfamiliar.

Automated EDA runs a repeatable set of summaries and visualizations across many columns. It is useful for a fast first pass, consistent reporting, and finding areas that deserve closer inspection. Some tools also compare subsets or generate shareable HTML reports.

A productive loop combines both: use an automated report to find a signal, then examine the underlying rows and domain context manually. A warning is a prompt to investigate, not an instruction to drop, impute, or transform a column.

## What Automated EDA Usually Checks

An EDA report organizes a first pass over the dataset. Its summaries are prompts for investigation, not verdicts about what to delete or transform.

| Report area | Questions to ask |
|---|---|
| Shape and data types | How many rows and columns are present? Are dates, categories, booleans, and numeric measurements represented correctly? |
| Missingness | Which columns and rows are incomplete? Could missingness itself carry meaning, and is it concentrated in a subgroup? |
| Cardinality and constants | Are there unique identifiers, near-unique fields, constant columns, or categories with very few observations? |
| Distributions | Are numeric features skewed, bounded, multimodal, or dominated by unusual values? Are category frequencies plausible? |
| Duplicates and validity | Are rows duplicated? Do values violate known ranges, units, or logical relationships? |
| Relationships | Are features strongly associated? Does a feature suspiciously reveal the target or a future outcome? |
| Target | For classification, are classes imbalanced? For regression, are values skewed, censored, or unusually concentrated? |

Automated reports often summarize these areas with descriptive statistics, plots, missingness tables, correlation measures, and warnings. The method used for correlation may depend on the data types and tool configuration; a single correlation number does not describe every relationship.

## Example: Investigate a Suspicious Feature

Suppose a churn report shows that `cancellation_reason` is almost perfectly associated with the target `churned`. That may look like an excellent predictor, but first ask when the reason is recorded. If it is entered only after a customer cancels, it is unavailable when the model is supposed to make an early-retention prediction. Including it would leak future information into training and produce an unrealistically optimistic evaluation.

The same report might flag `customer_id` as a nearly unique column. A unique identifier can help join tables, but usually does not describe a reusable customer pattern and can encourage memorization. Do not automatically remove either column: investigate their definitions, timestamps, and intended prediction setting, then document the decision.

## Common Automated EDA Tools

Different tools overlap, but they do not all solve the same problem:

- `ydata-profiling` generates a report with column summaries, distributions, missingness, correlations, and alerts. It was formerly known as pandas-profiling.
- `Sweetviz` creates visual reports and can compare datasets or target-defined groups.
- `AutoViz` automates a selection of plots to help scan relationships and distributions.
- `D-Tale` provides an interactive interface for inspecting tabular data and generated summaries.
- `Great Expectations` focuses on data validation: users define expectations, such as allowed ranges or required values, and check whether data satisfies them.

Profiling tools summarize what is present. Validation tools check data against explicit rules. A profiling alert may suggest a validation rule, but a report by itself does not establish that a value is invalid. Tool capabilities, defaults, and supported data types change by version, so check the current documentation before relying on a specific report feature.

## How EDA Findings Affect Modeling

EDA is useful when each finding leads to a testable question or a deliberate modeling decision.

- High missingness may motivate investigating why values are absent, adding an indicator, imputing, or excluding a feature. The appropriate choice depends on the data-generating process and the model.
- A constant or near-constant feature often contributes little, but first verify that it is not a status flag or a column whose rare value is important.
- Strongly correlated features may be redundant for some models, but correlation alone does not make a feature invalid. Tree models and linear models can react differently to correlated inputs.
- Class imbalance can make accuracy misleading. Inspect class counts and choose metrics such as precision, recall, F1, PR-AUC, or ROC-AUC according to the use case.
- Differences between training and test distributions can indicate population shift, collection changes, or an inappropriate split. They do not by themselves identify the cause.
- Suspiciously predictive columns deserve a leakage check: ask whether the feature is truly available at prediction time and whether it was produced using the target or future information.

EDA findings generate hypotheses. Verify them with source data, domain knowledge, and appropriate validation before changing features or evaluation design.

## Limitations and Risks

Automated EDA can reveal patterns, but it cannot infer the data's real-world meaning or decide whether a pattern is valid.

- Correlation does not establish causation, and many pairwise comparisons can surface chance associations.
- An outlier may be a measurement error, a rare but valid case, or the most important case for the application.
- A report may miss domain-specific impossibilities, collection artifacts, temporal structure, or leakage that requires understanding when fields become available.
- Summaries can hide rare subgroups. A small but important population may be invisible in overall plots or aggregate statistics.
- Very wide or large datasets can make full reports slow, memory-intensive, or difficult to interpret. Sampling may help but can omit rare patterns.
- Reports can expose sensitive data through examples, distributions, or generated files. Follow access controls, privacy requirements, and data-retention rules before sharing or storing them.

An automated report is not a substitute for data documentation, validation rules, or expert review.

## When to Use Automated EDA

Use automated EDA early when receiving an unfamiliar dataset, after a meaningful data refresh, or when you need a consistent first-pass report across many columns. It can help prioritize manual investigation before feature engineering and model selection.

It is also useful for comparing datasets or populations, such as training and serving data. Make comparisons meaningful: account for time, sampling, and how each split was created rather than treating any distribution difference as proof of a problem.

Do not rely on it as the only review for high-stakes data, unusual domains, or final feature-selection decisions. Combine reports with manual inspection, domain expertise, and explicit validation checks.

## How Automated EDA Fits into the ML Workflow

A responsible first-pass workflow is:

1. **Define the prediction problem.** Identify the unit represented by each row, the target, the prediction time, and which information would be available then.
2. **Check the data contract.** Confirm schema, types, units, provenance, and expected ranges with the data owner or documentation.
3. **Create the evaluation split.** Respect time, groups, or other dependencies; fit data-dependent preprocessing only on training data.
4. **Generate and review a profile.** Inspect shape, types, missingness, distributions, cardinality, duplicates, and target relationships. Keep sensitive reports access-controlled.
5. **Investigate findings.** Trace alerts to underlying records, domain rules, and feature timestamps. Decide whether to clean, transform, retain, exclude, or monitor each issue.
6. **Record decisions and validate.** Document assumptions and checks, then test changes in a modeling pipeline and evaluate with an appropriate validation strategy.

Automated EDA belongs near the beginning of the workflow, but it can be repeated after transformations, on new data, and in monitoring. It speeds up inspection; responsibility for interpreting evidence and choosing actions remains with the practitioner.